# HypoSpace — Exploratory Notebook

Interactive exploration of the HypoSpace interpretability pipeline on real GPT-2 activations.

**Pipeline:**
```
GPT-2 → NNSightExtractor → ActivationPreprocessor
      → HierarchyEngine  (top-k features)
      → SemanticInterpreter (intensity labels)
      → FaithfulnessChecker (governance scorecard)
      → SemanticCanvas  (visualization)
```

**Sections:**
0. Diagnostics — verify subsystem health  
1. Synthetic warm-up — understand the API on a small example  
2. GPT-2 extraction — real activations from multiple layers  
3. Cross-layer feature analysis — compare h.0 / h.6 / h.11  
4. Governance deep-dive — faithfulness, stability, threshold experiments  
5. Semantic canvas — interactive scatter plot with nearest-neighbor edges  
6. Kernel library — versioning and cross-run concept matching  

In [1]:
# Run once per environment
!pip install -q jupyter plotly
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q nnsight pyvene diskcache

In [2]:
import sys
import os
import warnings
import shutil
import tempfile
import pathlib
import statistics
from itertools import product

sys.path.insert(0, os.path.abspath('.'))

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from api import HypoSpaceAPI
from core.config import DecoderConfig, RuntimeConfig, GovernanceConfig
from viz.canvas import SemanticCanvas
from diagnostics import run_diagnostics

try:
    from data.nnsight_extractor import NNSightExtractor
    NNSIGHT_AVAILABLE = True
    print('nnsight available')
except ImportError:
    NNSIGHT_AVAILABLE = False
    print('nnsight not available — install torch+nnsight to run GPT-2 sections')

api = HypoSpaceAPI(config=DecoderConfig(top_k=8, runtime=RuntimeConfig(device='cpu')))
canvas = SemanticCanvas()
print('HypoSpaceAPI initialized')

nnsight available


HypoSpaceAPI initialized


## Section 0 — Diagnostics

Verify that every subsystem is healthy before running the rest of the notebook.

In [3]:
report = run_diagnostics()
print(f'Overall status: {report.overall_status}')
print()
for probe in report.probes:
    icon = {'ok': 'OK  ', 'degraded': 'WARN', 'error': 'ERR '}.get(probe.status, '?   ')
    print(f'  [{icon}] {probe.subsystem:32s} ({probe.latency_ms:.1f} ms)')
print()
print('Dependencies:')
for dep in report.dependencies:
    icon = 'yes' if dep.available else 'NO '
    print(f'  [{icon}] {dep.name:15s}  {dep.note or ""}')

/home/user/HypoSpace/api.py:109: UserWarning: No prior kernel found for 'diag-model-diag-layer'; cross-run match rate set to 0.0
  result = self.decoder.decode(model_name=model_name, layer=layer, activations=values, version=version)


Overall status: degraded

  [OK  ] config                           (0.0 ms)
  [OK  ] preprocessor                     (0.0 ms)
  [OK  ] hierarchy                        (0.0 ms)
  [OK  ] kernel_library                   (1.0 ms)
  [OK  ] extractor                        (5.8 ms)
  [OK  ] semantic                         (0.1 ms)
  [WARN] mechanistic                      (0.0 ms)
  [OK  ] full_pipeline                    (12.4 ms)
  [OK  ] governance                       (0.0 ms)
  [OK  ] nnsight                          (3293.0 ms)
  [OK  ] pyvene                           (2274.7 ms)
  [OK  ] sae_backend                      (0.1 ms)

Dependencies:
  [yes] torch            
  [yes] nnsight          
  [yes] pyvene           
  [yes] diskcache        
  [yes] sae_backend      


## Section 1 — Synthetic Warm-Up

Before hitting GPT-2, understand what the system does on a small, inspectable example.  
We pass an 8-dimensional activation vector and walk through every output.

In [4]:
DEMO_ACTIVATIONS = [0.9, -0.1, 0.4, 0.85, 0.2, -0.7, 0.05, 0.6]

result_warmup = api.decode_and_score('warmup', 'layer_0', DEMO_ACTIVATIONS, version='0.1.0')

print('Input activations (raw, 8-dim):')
for i, v in enumerate(DEMO_ACTIVATIONS):
    bar = chr(9608) * int(abs(v) * 20)
    sign = '+' if v >= 0 else ''
    print(f'  [{i}] {sign}{v:.2f}  {bar}')

print(f'\nTop-{len(result_warmup.decode.features)} features (magnitude top-k, normalized):')
for f in result_warmup.decode.features:
    print(f'  {f.id:38s} score={f.score:.4f}  {f.label}')

sc = result_warmup.scorecard
print('\nGovernance scorecard:')
print(f'  faithfulness  = {sc.faithfulness_score:.4f}  (mean ablation effect size)')
print(f'  stability     = {sc.stability_score:.4f}  (variance-based)')
print(f'  risk_flag     = {sc.risk_flag}')
print(f'  method        = {sc.intervention_method}')

Input activations (raw, 8-dim):
  [0] +0.90  ██████████████████
  [1] -0.10  ██
  [2] +0.40  ████████
  [3] +0.85  █████████████████
  [4] +0.20  ████
  [5] -0.70  ██████████████
  [6] +0.05  █
  [7] +0.60  ████████████

Top-8 features (magnitude top-k, normalized):
  layer_0:feature:0                      score=1.0000  high-intensity concept around activation index 0
  layer_0:feature:1                      score=0.9444  high-intensity concept around activation index 3
  layer_0:feature:2                      score=0.7778  medium-intensity concept around activation index 5
  layer_0:feature:3                      score=0.6667  medium-intensity concept around activation index 7
  layer_0:feature:4                      score=0.4444  medium-intensity concept around activation index 2
  layer_0:feature:5                      score=0.2222  low-intensity concept around activation index 4
  layer_0:feature:6                      score=0.1111  low-intensity concept around activation index 1
 

In [5]:
features = result_warmup.decode.features
color_map = {'high': '#e74c3c', 'medium': '#f39c12', 'low': '#3498db'}
bar_colors = [color_map.get(f.label.split('-')[0], '#95a5a6') for f in features]

fig = go.Figure(go.Bar(
    x=[f'dim {f.source_index}' for f in features],
    y=[f.score for f in features],
    marker_color=bar_colors,
    text=[f'{f.score:.3f}' for f in features],
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>score: %{y:.4f}<extra></extra>',
))
fig.update_layout(
    title='Synthetic Warm-Up: Top-8 Feature Scores',
    xaxis_title='Activation Dimension',
    yaxis_title='Normalized Score',
    yaxis_range=[0, 1.15],
    height=400,
    showlegend=False,
    annotations=[dict(
        x=0.01, y=0.97, xref='paper', yref='paper',
        text='red = high   orange = medium   blue = low intensity',
        showarrow=False, font=dict(size=11),
    )],
)
fig.show()

## Section 2 — Live GPT-2 Extraction

Extract real activations from GPT-2 using `NNSightExtractor`.  
We pull from three layers in a single forward pass: early (h.0), middle (h.6), and late (h.11).  
Activations are cached under `.hypo_cache/nnsight/` — repeated runs skip the forward pass.

In [6]:
PROMPTS = {
    'capital': 'The capital of France is',
    'code':    'def fibonacci(n):',
    'story':   'Once upon a time in a land',
    'math':    'The answer to 1 + 1 is',
}
LAYERS = ['transformer.h.0', 'transformer.h.6', 'transformer.h.11']

if not NNSIGHT_AVAILABLE:
    print('Install torch+nnsight to run this section')
else:
    extractor = NNSightExtractor('gpt2', device='cpu', cache_dir='.hypo_cache')

    sample_prompt = PROMPTS['capital']
    print(f"Extracting from GPT-2: '{sample_prompt}'")
    print(f'Layers: {LAYERS}')
    print()

    layer_activations = extractor.extract_layers(sample_prompt, layer_paths=LAYERS)

    for lp, acts in layer_activations.items():
        short = lp.replace('transformer.', '')
        print(f'  {short:8s}  dim={len(acts)}  '
              f'min={min(acts):.3f}  max={max(acts):.3f}  '
              f'mean={statistics.mean(acts):.4f}  '
              f'std={statistics.stdev(acts):.4f}')

Extracting from GPT-2: 'The capital of France is'
Layers: ['transformer.h.0', 'transformer.h.6', 'transformer.h.11']



config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

  h.0       dim=768  min=-26.569  max=28.771  mean=0.0321  std=2.0400
  h.6       dim=768  min=-25.259  max=44.441  mean=0.0825  std=3.0815
  h.11      dim=768  min=-173.858  max=187.628  mean=0.0495  std=15.5118


In [7]:
if not NNSIGHT_AVAILABLE:
    print('Skipping - nnsight not available')
else:
    short_names = [lp.replace('transformer.', '') for lp in LAYERS]
    fig = make_subplots(rows=1, cols=len(LAYERS), subplot_titles=short_names)

    for col, lp in enumerate(LAYERS, 1):
        acts = layer_activations[lp]
        fig.add_trace(
            go.Histogram(x=acts, nbinsx=60, marker_color='#3498db',
                         opacity=0.75, showlegend=False),
            row=1, col=col
        )

    fig.update_layout(
        title_text=f'GPT-2 Activation Distributions  "{sample_prompt}"',
        height=380,
    )
    fig.update_xaxes(title_text='Activation Value')
    fig.update_yaxes(title_text='Count', col=1)
    fig.show()

## Section 3 — Cross-Layer Feature Analysis

Decode all 4 prompts × 3 layers using `decode_and_score()` on the cached activations.  
This uses Path A (raw activations) with the stub intervention — fast and CPU-only.  
Results are stored in `results[prompt_key][layer_path]`.

In [8]:
if not NNSIGHT_AVAILABLE:
    print('Skipping - nnsight not available')
else:
    print('Extracting activations for all prompts...')
    raw = {}
    for pkey, prompt in PROMPTS.items():
        raw[pkey] = extractor.extract_layers(prompt, layer_paths=LAYERS)
        print(f'  {pkey}: {len(LAYERS)} layers')

    print('\nDecoding features...')
    results = {}
    for pkey in PROMPTS:
        results[pkey] = {}
        for lp in LAYERS:
            layer_label = lp.replace('transformer.h.', 'h')
            results[pkey][lp] = api.decode_and_score(
                'gpt2', layer_label, raw[pkey][lp], version='0.1.0'
            )

    n = len(PROMPTS) * len(LAYERS)
    print(f'Done: {len(PROMPTS)} prompts x {len(LAYERS)} layers = {n} results')

Extracting activations for all prompts...
  capital: 3 layers
  code: 3 layers
  story: 3 layers
  math: 3 layers

Decoding features...
Done: 4 prompts x 3 layers = 12 results


/home/user/HypoSpace/api.py:109: UserWarning: No prior kernel found for 'gpt2-h0'; cross-run match rate set to 0.0
  result = self.decoder.decode(model_name=model_name, layer=layer, activations=values, version=version)
/home/user/HypoSpace/api.py:109: UserWarning: No prior kernel found for 'gpt2-h6'; cross-run match rate set to 0.0
  result = self.decoder.decode(model_name=model_name, layer=layer, activations=values, version=version)
/home/user/HypoSpace/api.py:109: UserWarning: No prior kernel found for 'gpt2-h11'; cross-run match rate set to 0.0
  result = self.decoder.decode(model_name=model_name, layer=layer, activations=values, version=version)


In [9]:
if not NNSIGHT_AVAILABLE:
    print('Skipping - nnsight not available')
else:
    TOP_K = 8
    pkey = 'capital'
    rank_labels = [f'Rank {i+1}' for i in range(TOP_K)]
    layer_labels = [lp.replace('transformer.', '') for lp in LAYERS]
    z_scores = []

    for lp in LAYERS:
        feats = sorted(results[pkey][lp].decode.features, key=lambda f: -f.score)
        row = [f.score for f in feats]
        row += [0.0] * (TOP_K - len(row))
        z_scores.append(row[:TOP_K])

    fig = go.Figure(go.Heatmap(
        z=z_scores,
        x=rank_labels,
        y=layer_labels,
        colorscale='Plasma',
        text=[[f'{v:.3f}' for v in row] for row in z_scores],
        texttemplate='%{text}',
        hovertemplate='<b>%{y} - %{x}</b><br>score: %{z:.4f}<extra></extra>',
    ))
    fig.update_layout(
        title=f'Feature Score by Rank  "{PROMPTS[pkey]}"',
        xaxis_title='Feature Rank',
        yaxis_title='Layer',
        height=320,
    )
    fig.show()

In [10]:
if not NNSIGHT_AVAILABLE:
    print('Skipping - nnsight not available')
else:
    target_layer = 'transformer.h.6'
    fig = go.Figure()

    for pkey in PROMPTS:
        feats = sorted(results[pkey][target_layer].decode.features, key=lambda f: -f.score)
        fig.add_trace(go.Bar(
            name=pkey,
            x=[f'dim {f.source_index}' for f in feats],
            y=[f.score for f in feats],
        ))

    fig.update_layout(
        barmode='group',
        title='Top Feature Dimensions per Prompt  GPT-2 h.6',
        xaxis_title='Activation Dimension',
        yaxis_title='Normalized Score',
        height=440,
    )
    fig.show()

## Section 4 — Governance Deep-Dive

Examine faithfulness and stability scores across all prompt/layer combinations.  
Then explore how the pass rate changes as the governance threshold tightens.

In [11]:
if not NNSIGHT_AVAILABLE:
    print('Skipping - nnsight not available')
else:
    prompt_col, layer_col = [], []
    faith_col, stab_col, risk_col, passes_col = [], [], [], []
    row_colors = []

    for pkey, lp in product(PROMPTS, LAYERS):
        sc = results[pkey][lp].scorecard
        prompt_col.append(pkey)
        layer_col.append(lp.replace('transformer.', ''))
        faith_col.append(f'{sc.faithfulness_score:.4f}')
        stab_col.append(f'{sc.stability_score:.4f}')
        risk_col.append(sc.risk_flag)
        passes_col.append('Y' if sc.passes_thresholds else 'N')
        row_colors.append('#d5f5e3' if sc.passes_thresholds else '#fadbd8')

    fig = go.Figure(go.Table(
        header=dict(
            values=['Prompt', 'Layer', 'Faithfulness', 'Stability', 'Risk Flag', 'Passes'],
            fill_color='steelblue',
            font=dict(color='white', size=13),
            align='center',
        ),
        cells=dict(
            values=[prompt_col, layer_col, faith_col, stab_col, risk_col, passes_col],
            fill_color=[row_colors] * 6,
            align='center',
            font=dict(size=12),
        ),
    ))
    fig.update_layout(
        title='Governance Scorecards  All Prompts x Layers',
        height=520,
    )
    fig.show()

In [12]:
if not NNSIGHT_AVAILABLE:
    print('Skipping - nnsight not available')
else:
    rows = []
    for pkey, lp in product(PROMPTS, LAYERS):
        sc = results[pkey][lp].scorecard
        short_lp = lp.replace('transformer.h.', 'h')
        rows.append({
            'label': pkey + '/' + short_lp,
            'faithfulness': sc.faithfulness_score,
            'stability': sc.stability_score,
            'risk': sc.risk_flag,
        })

    fig = px.scatter(
        rows,
        x='faithfulness', y='stability',
        color='risk',
        text='label',
        color_discrete_map={
            'ok': '#27ae60',
            'low_faithfulness': '#e74c3c',
            'low_stability': '#f39c12',
        },
        title='Governance: Faithfulness vs Stability  (all prompts x layers)',
        height=540,
    )
    fig.update_traces(textposition='top center', marker=dict(size=12))

    cfg = GovernanceConfig()
    fig.add_hline(y=cfg.min_stability_score, line_dash='dash', line_color='orange',
                  annotation_text=f'min_stability={cfg.min_stability_score}')
    fig.add_vline(x=cfg.min_faithfulness_score, line_dash='dash', line_color='red',
                  annotation_text=f'min_faithfulness={cfg.min_faithfulness_score}')
    fig.update_layout(xaxis_range=[0, 1.05], yaxis_range=[0, 1.05])
    fig.show()

In [13]:
if not NNSIGHT_AVAILABLE:
    print('Skipping - nnsight not available')
else:
    thresholds = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.8, 0.9, 0.95, 1.0]
    total = len(PROMPTS) * len(LAYERS)

    pass_rates = [
        sum(
            1 for pkey in PROMPTS for lp in LAYERS
            if (results[pkey][lp].scorecard.faithfulness_score >= t
                and results[pkey][lp].scorecard.stability_score >= t)
        ) / total
        for t in thresholds
    ]

    cfg = GovernanceConfig()
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=thresholds,
        y=pass_rates,
        mode='lines+markers',
        line=dict(color='#3498db', width=2),
        marker=dict(size=8),
        hovertemplate='threshold=%{x:.2f}<br>pass rate=%{y:.1%}<extra></extra>',
    ))
    fig.add_vline(x=cfg.min_faithfulness_score, line_dash='dash', line_color='red',
                  annotation_text=f'default ({cfg.min_faithfulness_score})')
    fig.update_layout(
        title='Governance Pass Rate vs Threshold  (faithfulness AND stability)',
        xaxis_title='Threshold applied to both metrics',
        yaxis_title='Fraction of Results Passing',
        yaxis_tickformat='.0%',
        yaxis_range=[-0.05, 1.1],
        height=420,
    )
    fig.show()

## Section 5 — Semantic Canvas Visualization

`SemanticCanvas.to_layout()` places each feature in 2D:  
- **x** = rank-normalized (rightmost = highest score)  
- **y** = score-normalized  

`to_edges()` links nearest neighbors by score proximity.  
Node size and color both encode feature score.

In [14]:
if not NNSIGHT_AVAILABLE:
    print('Skipping - nnsight not available')
else:
    demo_res   = results['capital']['transformer.h.6']
    layout_pts = canvas.to_layout(demo_res.decode.features)
    edge_list  = canvas.to_edges(demo_res.decode.features)

    fig = go.Figure()

    # Nearest-neighbor edges (behind nodes)
    for src_id, tgt_id, strength in edge_list:
        src = next((p for p in layout_pts if p['id'] == src_id), None)
        tgt = next((p for p in layout_pts if p['id'] == tgt_id), None)
        if src and tgt:
            alpha = min(strength, 0.85)
            fig.add_trace(go.Scatter(
                x=[src['x'], tgt['x'], None],
                y=[src['y'], tgt['y'], None],
                mode='lines',
                line=dict(color=f'rgba(100,100,200,{alpha:.2f})', width=2),
                showlegend=False,
                hoverinfo='skip',
            ))

    # Feature nodes
    fig.add_trace(go.Scatter(
        x=[p['x'] for p in layout_pts],
        y=[p['y'] for p in layout_pts],
        mode='markers+text',
        text=[p['label'] or p['id'] for p in layout_pts],
        textposition='top center',
        marker=dict(
            size=[12 + p['score'] * 24 for p in layout_pts],
            color=[p['score'] for p in layout_pts],
            colorscale='Plasma',
            showscale=True,
            colorbar=dict(title='Score', thickness=14),
            line=dict(width=1, color='white'),
        ),
        customdata=[[p['id'], p['score']] for p in layout_pts],
        hovertemplate=('<b>%{customdata[0]}</b><br>'
                       'score: %{customdata[1]:.4f}<br>'
                       'label: %{text}<extra></extra>'),
        showlegend=False,
    ))

    fig.update_layout(
        title='Semantic Canvas  "The capital of France is" @ h.6',
        xaxis=dict(title='Rank (higher score = right)', showgrid=False, zeroline=False),
        yaxis=dict(title='Score (normalized)', showgrid=False, zeroline=False),
        height=540,
    )
    fig.show()

## Section 6 — Kernel Library & Cross-Run Matching

The `KernelLibrary` persists versioned feature snapshots and computes `cross_run_match_rate`:  
how many top features by `source_index` reappear across versions.

We save three versions at h.6:
- v0.1.0 and v0.2.0 — same prompt → expect a high match rate (stable concepts)
- v0.1.0 and v0.3.0 — different prompt → expect a lower match rate

In [15]:
if not NNSIGHT_AVAILABLE:
    print('Skipping - nnsight not available')
else:
    tmp_dir = pathlib.Path(tempfile.mkdtemp()) / 'kern_demo'
    tmp_dir.mkdir(parents=True, exist_ok=True)

    api_kern = HypoSpaceAPI(config=DecoderConfig(
        top_k=6,
        runtime=RuntimeConfig(cache_dir=str(tmp_dir))
    ))

    acts_capital = raw['capital']['transformer.h.6']
    acts_code    = raw['code']['transformer.h.6']

    r_v1 = api_kern.decode_and_score('gpt2', 'h6', acts_capital, version='0.1.0')
    r_v2 = api_kern.decode_and_score('gpt2', 'h6', acts_capital, version='0.2.0')
    r_v3 = api_kern.decode_and_score('gpt2', 'h6', acts_code,    version='0.3.0')

    match_same = float(r_v2.decode.metadata.get('cross_run_match_rate', '0'))
    match_diff = float(r_v3.decode.metadata.get('cross_run_match_rate', '0'))

    print('Cross-run concept matching @ h.6:')
    print(f'  Same prompt  0.1.0 -> 0.2.0 : {match_same:.3f}')
    print(f'  Diff prompt  0.1.0 -> 0.3.0 : {match_diff:.3f}')
    print()
    print('v0.1.0 features (capital):')
    for f in r_v1.decode.features:
        print(f'  dim {f.source_index:4d}  score={f.score:.4f}  {f.label}')
    print()
    print('v0.3.0 features (code):')
    for f in r_v3.decode.features:
        print(f'  dim {f.source_index:4d}  score={f.score:.4f}  {f.label}')

    shutil.rmtree(str(tmp_dir.parent), ignore_errors=True)

Cross-run concept matching @ h.6:
  Same prompt  0.1.0 -> 0.2.0 : 1.000
  Diff prompt  0.1.0 -> 0.3.0 : 0.667

v0.1.0 features (capital):
  dim  447  score=1.0000  high-intensity concept around activation index 447
  dim  373  score=0.5684  medium-intensity concept around activation index 373
  dim   64  score=0.5522  medium-intensity concept around activation index 64
  dim  481  score=0.4800  medium-intensity concept around activation index 481
  dim  393  score=0.4053  medium-intensity concept around activation index 393
  dim  266  score=0.3688  low-intensity concept around activation index 266

v0.3.0 features (code):
  dim  447  score=1.0000  high-intensity concept around activation index 447
  dim  373  score=0.5783  medium-intensity concept around activation index 373
  dim   64  score=0.5576  medium-intensity concept around activation index 64
  dim  266  score=0.3728  low-intensity concept around activation index 266
  dim  640  score=0.3122  low-intensity concept around acti

/home/user/HypoSpace/api.py:109: UserWarning: No prior kernel found for 'gpt2-h6'; cross-run match rate set to 0.0
  result = self.decoder.decode(model_name=model_name, layer=layer, activations=values, version=version)


In [16]:
if not NNSIGHT_AVAILABLE:
    print('Skipping - nnsight not available')
else:
    pair_labels = ['same prompt (0.1->0.2)', 'diff prompt (0.1->0.3)']
    pair_values = [match_same, match_diff]
    pair_colors = ['#27ae60' if v >= 0.7 else '#e74c3c' if v < 0.3 else '#f39c12'
                   for v in pair_values]

    fig = go.Figure(go.Bar(
        x=pair_labels,
        y=pair_values,
        marker_color=pair_colors,
        text=[f'{v:.3f}' for v in pair_values],
        textposition='outside',
        hovertemplate='%{x}<br>match rate: %{y:.3f}<extra></extra>',
    ))
    fig.update_layout(
        title='Kernel Concept Consistency Across Runs @ h.6',
        xaxis_title='Run Pair',
        yaxis_title='Cross-Run Match Rate',
        yaxis_range=[0, 1.15],
        height=420,
        showlegend=False,
    )
    fig.show()